# CrossGuard Remote GPU Worker
### High-Performance YOLOv8n CUDA Inference on Google Colab

This notebook turns a free or Pro Google Colab instance into a dedicated, hardware-accelerated CUDA perception worker for the **CrossGuard** Adaptive Traffic Management Research Simulator.

**Architecture Role**:
- Runs YOLOv8n on an NVIDIA T4 / A100 GPU using PyTorch CUDA.
- Exposes a secure, real-time HTTP tunnel via Cloudflare (`trycloudflare.com`).
- Returns genuine bounding boxes, class labels, confidences, and measured GPU inference latency.
- Local CrossGuard manages ByteTrack tracking, camera homography, spatial analysis, and the adaptive FSM controller.

## 1. Verify CUDA Hardware Acceleration
Ensure your Colab runtime is set to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version:   {torch.__version__}")
print(f"CUDA Available:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Name:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total:        {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise SystemError("CUDA is not available! Switch runtime to GPU under Runtime -> Change runtime type.")
print("=" * 60)

## 2. Install Dependencies
Installs FastAPI, Uvicorn, Ultralytics YOLOv8, and pycloudflared for tunnel creation.

In [ ]:
!pip install -q fastapi uvicorn ultralytics opencv-python-headless pycloudflared pydantic python-multipart

## 3. Initialize YOLOv8n on CUDA

In [ ]:
import numpy as np
import cv2
from ultralytics import YOLO

print("[*] Loading yolov8n.pt onto CUDA...")
model = YOLO("yolov8n.pt")
# Warm up CUDA kernels
dummy_frame = np.zeros((640, 640, 3), dtype=np.uint8)
_ = model.predict(source=dummy_frame, device="cuda", verbose=False)
print("[+] YOLOv8n initialized and warmed up on CUDA successfully!")

## 4. Define CrossGuard Worker FastAPI Server

In [ ]:
import base64
import time
from typing import List, Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

app = FastAPI(title="CrossGuard Colab CUDA Worker", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

class DetectionItemModel(BaseModel):
    x1: float
    y1: float
    x2: float
    y2: float
    confidence: float
    class_id: int
    class_name: str

class InferenceRequestModel(BaseModel):
    image_b64: str
    confidence_threshold: float = 0.35
    iou_threshold: float = 0.5
    frame_index: Optional[int] = None

class InferenceResponseModel(BaseModel):
    status: str = "ok"
    detections: List[DetectionItemModel] = []
    inference_time_ms: float = 0.0
    device: str = "cuda"
    gpu_name: str = GPU_NAME
    frame_index: Optional[int] = None

class HealthResponseModel(BaseModel):
    status: str = "ok"
    cuda_available: bool = True
    gpu_name: str = GPU_NAME
    gpu_memory_total_mb: Optional[float] = None
    gpu_memory_allocated_mb: Optional[float] = None
    yolo_loaded: bool = True
    model_name: str = "yolov8n.pt"
    torch_version: str = torch.__version__
    device: str = "cuda"

@app.get("/health", response_model=HealthResponseModel)
def health():
    vram_tot = round(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024), 1)
    vram_alc = round(torch.cuda.memory_allocated(0) / (1024 * 1024), 1)
    return HealthResponseModel(
        cuda_available=torch.cuda.is_available(),
        gpu_name=GPU_NAME,
        gpu_memory_total_mb=vram_tot,
        gpu_memory_allocated_mb=vram_alc,
    )

@app.post("/detect", response_model=InferenceResponseModel)
def detect(req: InferenceRequestModel):
    try:
        img_bytes = base64.b64decode(req.image_b64)
        np_arr = np.frombuffer(img_bytes, dtype=np.uint8)
        frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
        if frame is None:
            raise ValueError("Empty image frame")
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

    t0 = time.perf_counter()
    results = model.predict(source=frame, device="cuda", conf=req.confidence_threshold, iou=req.iou_threshold, verbose=False)
    t1 = time.perf_counter()
    latency_ms = round((t1 - t0) * 1000.0, 2)

    dets = []
    if results and len(results) > 0:
        res = results[0]
        if hasattr(res, "boxes") and res.boxes is not None and len(res.boxes) > 0:
            boxes = res.boxes
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            clses = boxes.cls.cpu().numpy()
            names = res.names if hasattr(res, "names") else {}
            for i in range(len(xyxy)):
                c_id = int(clses[i])
                dets.append(DetectionItemModel(
                    x1=float(xyxy[i][0]), y1=float(xyxy[i][1]),
                    x2=float(xyxy[i][2]), y2=float(xyxy[i][3]),
                    confidence=float(confs[i]),
                    class_id=c_id,
                    class_name=names.get(c_id, str(c_id)).lower(),
                ))

    return InferenceResponseModel(
        detections=dets,
        inference_time_ms=latency_ms,
        device="cuda",
        gpu_name=GPU_NAME,
        frame_index=req.frame_index,
    )

## 5. Launch Worker & Cloudflare Tunnel
Runs the FastAPI server locally in the background on port 8000 and exposes it via Cloudflare Tunnel.

In [ ]:
import atexit
import json
import os
import re
import threading
import time
import urllib.error
import urllib.request
import uvicorn
from pycloudflared import try_cloudflare

# 1. Start Uvicorn in background thread
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2.0)

# 2. Start Cloudflare Tunnel
print("[*] Starting Cloudflare tunnel to 127.0.0.1:8000...")
public_url = try_cloudflare(port=8000)
worker_endpoint = public_url.tunnel.rstrip("/")

# 3. Dynamic Registry Integration (GitHub Gist)
gist_id = os.environ.get("CROSSGUARD_GPU_REGISTRY_GIST_ID", "").strip()
github_token = os.environ.get("CROSSGUARD_GPU_REGISTRY_GITHUB_TOKEN", "").strip()
try:
    from google.colab import userdata
    if not gist_id:
        try:
            gist_id = str(userdata.get("CROSSGUARD_GPU_REGISTRY_GIST_ID") or "").strip()
        except Exception:
            pass
    if not github_token:
        try:
            github_token = str(userdata.get("CROSSGUARD_GPU_REGISTRY_GITHUB_TOKEN") or "").strip()
        except Exception:
            pass
except Exception:
    pass

registry_status = "not configured"

def update_gist_record(status="online"):
    if not gist_id or not github_token:
        return False
    m = re.search(r"([a-f0-9]{6,})", str(gist_id), re.I)
    clean_gid = m.group(1) if m else str(gist_id)
    record_dict = {
        "worker_url": worker_endpoint,
        "worker_id": "crossguard-colab",
        "gpu_name": GPU_NAME,
        "status": status,
        "updated_at": time.time(),
        "protocol_version": "1",
        "heartbeat_interval_s": 60.0,
    }
    record_json = json.dumps(record_dict, indent=2)
    payload = {
        "description": f"CrossGuard Remote GPU Worker Registry [{status.upper()}]",
        "files": {
            "crossguard_worker.json": {
                "content": record_json
            },
            "gistfile1.txt": {
                "content": record_json
            }
        }
    }
    req = urllib.request.Request(
        f"https://api.github.com/gists/{clean_gid}",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {github_token.strip()}",
            "Content-Type": "application/json",
            "User-Agent": "CrossGuard-Colab/1.0",
        },
        method="PATCH",
    )
    try:
        with urllib.request.urlopen(req, timeout=10.0) as resp:
            return resp.status in (200, 201)
    except urllib.error.HTTPError as e:
        err_body = e.read().decode("utf-8", errors="ignore")
        print(f"[-] Registry update HTTP Error {e.code}: {e.reason} - {err_body}")
        return False
    except Exception as e:
        print(f"[-] Registry update network error: {e}")
        return False

if gist_id and github_token:
    if update_gist_record(status="online"):
        registry_status = "updated"
        def _heartbeat():
            while True:
                time.sleep(60.0)
                try:
                    update_gist_record(status="online")
                except Exception:
                    pass
        threading.Thread(target=_heartbeat, daemon=True).start()
        atexit.register(lambda: update_gist_record(status="offline"))
    else:
        registry_status = "update failed"
else:
    if not gist_id:
        print("[-] Notice: CROSSGUARD_GPU_REGISTRY_GIST_ID not found in Colab Secrets or env.")
    if not github_token:
        print("[-] Notice: CROSSGUARD_GPU_REGISTRY_GITHUB_TOKEN not found in Colab Secrets or env.")

print("=" * 70)
print("CrossGuard GPU Worker")
print(f"GPU: {GPU_NAME}")
print(f"CUDA: {'available' if torch.cuda.is_available() else 'unavailable'}")
print(f"Worker URL: {worker_endpoint}")
print(f"Registry: {registry_status}")
print("Status: ONLINE")
print("=" * 70)
if registry_status != "updated":
    print("Manual fallback: Set CROSSGUARD_GPU_WORKER_URL=\"" + worker_endpoint + "\" in your .env")


## 6. Self-Test Health and Inference
Verify that the worker responds to local requests with measured CUDA latency.

In [ ]:
import requests

# Test /health
h_resp = requests.get("http://127.0.0.1:8000/health").json()
print("Health check:", h_resp)

# Test /detect with blank frame
_, buf = cv2.imencode(".jpg", dummy_frame)
b64 = base64.b64encode(buf).decode("ascii")
d_resp = requests.post("http://127.0.0.1:8000/detect", json={"image_b64": b64}).json()
print(f"Test Inference Latency: {d_resp['inference_time_ms']:.2f} ms on {d_resp['gpu_name']}")